# ADAPT-VQE vs Qubit ADAPT-VQE: Quantum Resource Comparison

This notebook compares Fermionic ADAPT-VQE and Qubit ADAPT-VQE algorithms, focusing on **quantum resource metrics** that matter for real hardware deployment.

## Key Metrics
- **CNOT Count vs Accuracy**: The "Money" metric - proves hardware efficiency
- **Circuit Depth**: Affects decoherence
- **Parameter Efficiency**: Optimization landscape complexity

**Goal**: Verify that Qubit-ADAPT-VQE reaches chemical accuracy ($1.6 \times 10^{-3}$ Hartree) with **fewer CNOT gates** than Fermionic ADAPT-VQE.

## 1. Setup and Imports

In [1]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import warnings
import logging
warnings.filterwarnings('ignore')

# Configure logging to see messages from the VQE algorithms
logging.basicConfig(
    level=logging.INFO,  # Set to INFO to see logger.info() messages
    format="%(asctime)s - %(name)s - %(levelname)s - %(message)s",
    datefmt="%H:%M:%S"
)

# Add framework to path
sys.path.insert(0, str(Path.cwd()))

import pennylane as qml
from core.hamiltonian_loader import HamiltonianLoader
from algorithms.vqe_adapt import AdaptVQE
from algorithms.vqe_qubit_adapt import QubitAdaptVQE
from core.backend_manager import BackendConfig

print("✓ Imports successful")
print("✓ Logging configured (INFO level)")

# Auto-reload modules during development (no kernel restart needed)
%load_ext autoreload
%autoreload 2


✓ Imports successful
✓ Logging configured (INFO level)


## 2. Configuration

In [45]:
# Algorithm parameters
MAX_OPERATORS = 20
GRADIENT_THRESHOLD = 1e-4
MAX_ITERATIONS = 100
OPTIMIZER = "COBYLA"

# Hamiltonian truncation
MAX_HAMILTONIAN_TERMS = 2000
TARGET_QUBITS = 8

# Chemical accuracy threshold
CHEMICAL_ACCURACY = 1.6e-3  # Hartree

# Molecules to test (use smaller molecules for faster comparison)
MOLECULES_TO_TEST = ['ala']  # Start with smaller amino acids

print(f"Configuration:")
print(f"  Max operators: {MAX_OPERATORS}")
print(f"  Gradient threshold: {GRADIENT_THRESHOLD}")
print(f"  Molecules: {MOLECULES_TO_TEST}")

Configuration:
  Max operators: 20
  Gradient threshold: 0.0001
  Molecules: ['ala']


## 3. Helper Functions

In [46]:
def get_cnot_count(qnode, params):
    """
    Extracts CNOT count from a PennyLane QNode using qml.specs.
    
    Args:
        qnode: PennyLane QNode
        params: Parameters for the circuit
    
    Returns:
        CNOT count (int)
    """
    try:
        specs = qml.specs(qnode)(params)
        # specs['resources'] is a Resources object, not a dict
        resources = specs.get('resources')
        if resources is not None:
            gate_types = resources.gate_types
            return gate_types.get('CNOT', 0)
        return 0
    except Exception as e:
        # Fallback: try accessing as dict if Resources object access fails
        try:
            specs = qml.specs(qnode)(params)
            gate_types = specs.get('resources', {}).get('gate_types', {})
            return gate_types.get('CNOT', 0)
        except:
            return 0

def get_circuit_depth(qnode, params):
    """
    Extracts circuit depth from a PennyLane QNode.
    
    Args:
        qnode: PennyLane QNode
        params: Parameters for the circuit
    
    Returns:
        Circuit depth (int)
    """
    try:
        specs = qml.specs(qnode)(params)
        return specs.get('depth', 0)
    except Exception as e:
        print(f"Warning: Could not get circuit depth: {e}")
        return 0

def count_cnots_from_operators(algorithm_instance):
    """
    Count CNOTs based on selected operators (fallback method).
    
    For ADAPT-VQE:
    - Single excitation: 2 CNOTs
    - Double excitation: 2 CNOTs (simplified)
    
    For Qubit ADAPT-VQE:
    - SingleExcitation: ~6 CNOTs
    - DoubleExcitation: ~8 CNOTs
    """
    if not hasattr(algorithm_instance, 'selected_operators'):
        return 0
    
    cnot_count = 0
    
    if isinstance(algorithm_instance, AdaptVQE):
        # Fermionic ADAPT-VQE
        for op_idx in algorithm_instance.selected_operators:
            if op_idx < len(algorithm_instance.operator_pool):
                op = algorithm_instance.operator_pool[op_idx]
                if op[0] == 'single':
                    cnot_count += 2  # RY-CNOT-RY-CNOT
                elif op[0] == 'double':
                    cnot_count += 2  # Simplified
    elif isinstance(algorithm_instance, QubitAdaptVQE):
        # Qubit ADAPT-VQE
        for op_idx in algorithm_instance.selected_operators:
            if op_idx < len(algorithm_instance.operator_pool):
                pauli_string = algorithm_instance.operator_pool[op_idx]
                # Count non-identity Pauli operators in the string
                non_i_count = sum(1 for c in pauli_string if c in ['X', 'Y', 'Z'])
                if non_i_count == 2:
                    cnot_count += 2  # 2-qubit Pauli string
                elif non_i_count == 4:
                    cnot_count += 6  # 4-qubit Pauli string
                else:
                    cnot_count += max(1, non_i_count - 1)
    
    return cnot_count

print("✓ Helper functions loaded")

✓ Helper functions loaded


## 4. Initialize Framework

In [47]:
# Initialize Hamiltonian loader
from config import DATASETS_DIR, MOLECULES_JSON

loader = HamiltonianLoader(
    hamiltonians_dir=DATASETS_DIR,
    molecules_json=MOLECULES_JSON
)

print(f"✓ Hamiltonian loader initialized")
print(f"Available molecules: {loader.list_molecules()[:10]}...")

23:30:32 - core.hamiltonian_loader - INFO - Loaded metadata for 13 molecules


✓ Hamiltonian loader initialized


AttributeError: 'HamiltonianLoader' object has no attribute 'list_molecules'

## 5. Run Comparison for Each Molecule

In [ ]:
from framework.algorithms import VanillaVQE


results = []

for molecule_abbrev in MOLECULES_TO_TEST:
    print(f"\n{'='*70}")
    print(f"Processing: {molecule_abbrev}")
    print(f"{'='*70}")
    
    try:
        # Load Hamiltonian
        hamiltonian = loader.load_hamiltonian(molecule_abbrev=molecule_abbrev)
        
        # Truncate if needed
        if hamiltonian.n_terms > MAX_HAMILTONIAN_TERMS:
            hamiltonian = hamiltonian.truncate(max_terms=1000, target_qubits=8)
        
        ref_energy = hamiltonian.molecule.reference_energy
        print(f"\nMolecule: {hamiltonian.molecule.name}")
        print(f"Qubits: {hamiltonian.n_qubits}")
        print(f"Hamiltonian terms: {hamiltonian.n_terms}")
        print(f"Reference energy: {ref_energy:.6f} Ha")
        
        # Backend configuration
        backend_config = BackendConfig(
            backend_type="statevector",
            n_qubits=hamiltonian.n_qubits
        )
        
        # ===== Run Fermionic ADAPT-VQE =====
        # print("\n>>> Running Fermionic ADAPT-VQE...")
        # adapt = AdaptVQE(
        #     hamiltonian=hamiltonian,
        #     max_operators=MAX_OPERATORS,
        #     gradient_threshold=GRADIENT_THRESHOLD,
        #     optimizer_name=OPTIMIZER,
        #     max_iterations=MAX_ITERATIONS,
        #     backend_config=backend_config
        # )
        # res_adapt = adapt.run()
        
        
        # print(f"  Energy: {res_adapt.calculated_energy:.8f} Ha")
        # print(f"  Error: {res_adapt.error:.6f} Ha")
        # print(f"  Operators selected: {res_adapt.n_iterations}")

        # ===== Run Qubit ADAPT-VQE =====
        print("\n>>> Running Qubit ADAPT-VQE...")
        q_adapt = QubitAdaptVQE(
            hamiltonian=hamiltonian,
            max_operators=MAX_OPERATORS,
            gradient_threshold=GRADIENT_THRESHOLD,
            optimizer_name=OPTIMIZER,
            max_iterations=MAX_ITERATIONS,
            backend_config=backend_config,
        )
        res_q_adapt = q_adapt.run()
        
        # Get CNOT count for Qubit ADAPT
        if res_q_adapt.optimal_parameters is not None and len(res_q_adapt.optimal_parameters) > 0:
            circuit_q_adapt = q_adapt._build_circuit(res_q_adapt.optimal_parameters)
            cnot_q_adapt = get_cnot_count(circuit_q_adapt, res_q_adapt.optimal_parameters)
            depth_q_adapt = get_circuit_depth(circuit_q_adapt, res_q_adapt.optimal_parameters)
            
            # Fallback to operator-based counting if specs fails
            if cnot_q_adapt == 0:
                cnot_q_adapt = count_cnots_from_operators(q_adapt)
        else:
            cnot_q_adapt = 0
            depth_q_adapt = 0
        
        print(f"  Energy: {res_q_adapt.calculated_energy:.8f} Ha")
        print(f"  Error: {res_q_adapt.error:.6f} Ha")
        print(f"  Operators selected: {res_q_adapt.n_iterations}")
        print(f"  CNOT count: {cnot_q_adapt}")
        
        # Store results
        results.append({
            'molecule': molecule_abbrev,
            'molecule_name': hamiltonian.molecule.name,
            'n_qubits': hamiltonian.n_qubits,
            'reference_energy': ref_energy,
            
            # Fermionic ADAPT results
            # 'adapt_energy': res_adapt.calculated_energy,
            # 'adapt_error': abs(res_adapt.error),
            # 'adapt_params': res_adapt.n_parameters,
            # 'adapt_operators': res_adapt.n_iterations,
            # 'adapt_cnots': cnot_adapt,
            # 'adapt_depth': depth_adapt,
            # 'adapt_runtime': res_adapt.runtime_seconds,
            
            # Qubit ADAPT results
            'qubit_adapt_energy': res_q_adapt.calculated_energy,
            'qubit_adapt_error': abs(res_q_adapt.error),
            'qubit_adapt_params': res_q_adapt.n_parameters,
            'qubit_adapt_operators': res_q_adapt.n_iterations,
            'qubit_adapt_cnots': cnot_q_adapt,
            'qubit_adapt_depth': depth_q_adapt,
            'qubit_adapt_runtime': res_q_adapt.runtime_seconds,
        })
        
    except Exception as e:
        print(f"\n❌ Error processing {molecule_abbrev}: {e}")
        import traceback
        traceback.print_exc()

print(f"\n{'='*70}")
print(f"Comparison complete! Processed {len(results)} molecules")
print(f"{'='*70}")

23:30:34 - core.hamiltonian_loader - INFO - Loading Hamiltonian from H5 file: /Users/byan2/Developer/qmprot_gse/framework/datasets/ala/ala.h5
23:30:34 - core.hamiltonian_loader - INFO - Early truncation: reading 1/23 chunks (estimated ~118,514 terms)



Processing: ala


23:30:34 - core.hamiltonian_loader - INFO - Loaded 112801 terms from ala.h5 (11 qubits)
23:30:34 - core.hamiltonian_loader - INFO - Truncating Hamiltonian from 112801 to max 1000 terms...
23:30:34 - core.hamiltonian_loader - INFO - Truncated to 1 terms on 10 qubits
23:30:35 - core.hamiltonian_loader - INFO - Ground state energy of 10-qubit truncated system: -186.451930
23:30:35 - algorithms.vqe_qubit_adapt - INFO - Running qubit_adapt_vqe on alanine
23:30:35 - core.backend_manager - INFO - Created device: lightning.qubit | n_qubits=10 | shots=None | type=statevector
23:30:35 - algorithms.vqe_qubit_adapt - INFO - Built FULL Pauli operator pool with 1770 operators
23:30:35 - algorithms.vqe_qubit_adapt - INFO - Gradient computation: n_qubits=10, n_electrons_used=5, HF state=|1111100000>, pool_size=1770
23:30:35 - algorithms.vqe_qubit_adapt - INFO -   Op 0 'Y0 X1': E+=-186.4519303050, E-=-186.4519303050, grad=0.00e+00
23:30:35 - algorithms.vqe_qubit_adapt - INFO -   Op 1 'X0 Y1': E+=-186.4


Molecule: alanine
Qubits: 10
Hamiltonian terms: 1
Reference energy: -317.691350 Ha

>>> Running Qubit ADAPT-VQE...


23:30:35 - algorithms.vqe_qubit_adapt - INFO -   Op 28 'Y1 X7': E+=-186.4519303050, E-=-186.4519303050, grad=0.00e+00
23:30:35 - algorithms.vqe_qubit_adapt - INFO -   Op 29 'X1 Y7': E+=-186.4519303050, E-=-186.4519303050, grad=0.00e+00
23:30:35 - algorithms.vqe_qubit_adapt - INFO -   Op 30 'Y1 X8': E+=-186.4519303050, E-=-186.4519303050, grad=0.00e+00
23:30:35 - algorithms.vqe_qubit_adapt - INFO -   Op 31 'X1 Y8': E+=-186.4519303050, E-=-186.4519303050, grad=0.00e+00
23:30:35 - algorithms.vqe_qubit_adapt - INFO -   Op 32 'Y1 X9': E+=-186.4519303050, E-=-186.4519303050, grad=0.00e+00
23:30:35 - algorithms.vqe_qubit_adapt - INFO -   Op 33 'X1 Y9': E+=-186.4519303050, E-=-186.4519303050, grad=0.00e+00
23:30:35 - algorithms.vqe_qubit_adapt - INFO -   Op 34 'Y2 X3': E+=-186.4519303050, E-=-186.4519303050, grad=0.00e+00
23:30:35 - algorithms.vqe_qubit_adapt - INFO -   Op 35 'X2 Y3': E+=-186.4519303050, E-=-186.4519303050, grad=0.00e+00
23:30:35 - algorithms.vqe_qubit_adapt - INFO -   Op 36 '

  Energy: -186.45193031 Ha
  Error: 131.239420 Ha
  Operators selected: 0
  CNOT count: 0

Comparison complete! Processed 1 molecules


## 6. Comparison Table

In [ ]:
if not results:
    print("No results to display.")
else:
    df = pd.DataFrame(results)
    
    print("\n" + "="*100)
    print(f"{'Metric':<30} | {'Fermionic ADAPT':<20} | {'Qubit ADAPT':<20} | {'Winner'}")
    print("-" * 100)
    
    for _, row in df.iterrows():
        mol = row['molecule']
        print(f"\nMolecule: {mol} ({row['molecule_name']})")
        print(f"{'Reference Energy':<30} | {row['reference_energy']:>20.6f} | {'':<20} |")
        print(f"{'Final Energy (Ha)':<30} | {row['adapt_energy']:>20.8f} | {row['qubit_adapt_energy']:>20.8f} |")
        print(f"{'Accuracy (|Error|)':<30} | {row['adapt_error']:>20.6e} | {row['qubit_adapt_error']:>20.6e} |")
        
        # Determine winner for accuracy
        if row['adapt_error'] < row['qubit_adapt_error']:
            acc_winner = "Fermionic"
        elif row['qubit_adapt_error'] < row['adapt_error']:
            acc_winner = "Qubit"
        else:
            acc_winner = "Tie"
        
        print(f"{'Parameters (Ops)':<30} | {row['adapt_params']:>20} | {row['qubit_adapt_params']:>20} |")
        print(f"{'Total CNOTs':<30} | {row['adapt_cnots']:>20} | {row['qubit_adapt_cnots']:>20} |", end="")
        
        # Determine winner for CNOT count
        if row['qubit_adapt_cnots'] < row['adapt_cnots']:
            cnot_winner = "✅ Qubit (WIN)"
        elif row['adapt_cnots'] < row['qubit_adapt_cnots']:
            cnot_winner = "Fermionic"
        else:
            cnot_winner = "Tie"
        print(f" {cnot_winner}")
        
        print(f"{'CNOTs per Parameter':<30} | {row['adapt_cnots']/max(row['adapt_params'],1):>20.1f} | {row['qubit_adapt_cnots']/max(row['qubit_adapt_params'],1):>20.1f} |")
        print(f"{'Circuit Depth':<30} | {row['adapt_depth']:>20} | {row['qubit_adapt_depth']:>20} |")
        print(f"{'Runtime (s)':<30} | {row['adapt_runtime']:>20.2f} | {row['qubit_adapt_runtime']:>20.2f} |")
        
        # Overall assessment
        print(f"\n{'Assessment':<30} |", end="")
        if row['qubit_adapt_cnots'] < row['adapt_cnots'] and row['qubit_adapt_error'] <= CHEMICAL_ACCURACY:
            print(f" ✅ SUCCESS: Qubit-ADAPT used fewer CNOTs to reach chemical accuracy!")
        elif row['qubit_adapt_cnots'] < row['adapt_cnots']:
            print(f" ⚠️  PARTIAL: Qubit-ADAPT used fewer CNOTs but energy may need improvement.")
        elif row['adapt_cnots'] < row['qubit_adapt_cnots']:
            print(f" ❌ Fermionic ADAPT used fewer CNOTs. Check convergence settings.")
        else:
            print(f" ➖ Similar CNOT counts.")
        
        print("-" * 100)

## 7. CNOT Count vs Accuracy Plot (The "Money" Metric)

In [ ]:
if not results:
    print("No results to plot.")
else:
    df = pd.DataFrame(results)
    
    fig, ax = plt.subplots(figsize=(12, 8))
    
    # Plot points for each molecule
    for _, row in df.iterrows():
        # Fermionic ADAPT
        ax.scatter(row['adapt_cnots'], row['adapt_error'], 
                  s=300, alpha=0.7, color='#ff7f0e', marker='o', 
                  edgecolors='black', linewidth=2, label='Fermionic ADAPT' if _ == 0 else '')
        ax.annotate(f"{row['molecule']}-F", 
                   (row['adapt_cnots'], row['adapt_error']),
                   xytext=(5, 5), textcoords='offset points', fontsize=9)
        
        # Qubit ADAPT
        ax.scatter(row['qubit_adapt_cnots'], row['qubit_adapt_error'], 
                  s=300, alpha=0.7, color='#2ca02c', marker='s', 
                  edgecolors='black', linewidth=2, label='Qubit ADAPT' if _ == 0 else '')
        ax.annotate(f"{row['molecule']}-Q", 
                   (row['qubit_adapt_cnots'], row['qubit_adapt_error']),
                   xytext=(5, -15), textcoords='offset points', fontsize=9)
    
    # Add chemical accuracy line
    ax.axhline(y=CHEMICAL_ACCURACY, color='red', linestyle='--', linewidth=2, 
               label=f'Chemical Accuracy ({CHEMICAL_ACCURACY:.1e} Ha)', zorder=0)
    ax.fill_between(ax.get_xlim(), 0, CHEMICAL_ACCURACY, alpha=0.1, color='green', 
                    label='Chemically Accurate Region', zorder=0)
    
    ax.set_xlabel('CNOT Count', fontsize=14, fontweight='bold')
    ax.set_ylabel('Energy Error |E - E_ref| [Hartree]', fontsize=14, fontweight='bold')
    ax.set_title('CNOT Count vs Accuracy: The "Money" Metric\n' +
                'Lower CNOT count at chemical accuracy = Better for hardware', 
                fontsize=16, fontweight='bold', pad=20)
    ax.set_yscale('log')
    ax.grid(True, alpha=0.3, linestyle='--')
    ax.legend(loc='upper right', fontsize=12, framealpha=0.9)
    
    # Add interpretation text
    textstr = ('Key Insight: Qubit-ADAPT should reach chemical accuracy\n' +
              'with fewer CNOTs than Fermionic-ADAPT.\n' +
              'This proves hardware-efficiency despite slower simulation.')
    props = dict(boxstyle='round', facecolor='wheat', alpha=0.8)
    ax.text(0.02, 0.98, textstr, transform=ax.transAxes, fontsize=11,
           verticalalignment='top', bbox=props)
    
    plt.tight_layout()
    plt.show()
    
    # Print summary
    print("\n" + "="*70)
    print("CNOT Count Summary")
    print("="*70)
    print(f"\nFermionic ADAPT-VQE:")
    print(f"  Mean CNOT count: {df['adapt_cnots'].mean():.1f}")
    print(f"  Mean error: {df['adapt_error'].mean():.6e} Ha")
    print(f"\nQubit ADAPT-VQE:")
    print(f"  Mean CNOT count: {df['qubit_adapt_cnots'].mean():.1f}")
    print(f"  Mean error: {df['qubit_adapt_error'].mean():.6e} Ha")
    
    cnot_reduction = ((df['adapt_cnots'].mean() - df['qubit_adapt_cnots'].mean()) / 
                     max(df['adapt_cnots'].mean(), 1) * 100)
    print(f"\nCNOT reduction: {cnot_reduction:.1f}% (Qubit vs Fermionic)")

## 8. Circuit Depth Comparison

In [ ]:
if not results:
    print("No results to plot.")
else:
    df = pd.DataFrame(results)
    
    fig, ax = plt.subplots(figsize=(10, 6))
    
    molecules = df['molecule'].tolist()
    x = np.arange(len(molecules))
    width = 0.35
    
    bars1 = ax.bar(x - width/2, df['adapt_depth'], width, label='Fermionic ADAPT-VQE', 
                   color='#ff7f0e', alpha=0.7, edgecolor='black', linewidth=1.5)
    bars2 = ax.bar(x + width/2, df['qubit_adapt_depth'], width, label='Qubit ADAPT-VQE', 
                   color='#2ca02c', alpha=0.7, edgecolor='black', linewidth=1.5)
    
    ax.set_xlabel('Molecule', fontsize=12, fontweight='bold')
    ax.set_ylabel('Circuit Depth', fontsize=12, fontweight='bold')
    ax.set_title('Circuit Depth Comparison', fontsize=14, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(molecules, rotation=45, ha='right')
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3, axis='y', linestyle='--')
    
    plt.tight_layout()
    plt.show()
    
    print(f"\nMean circuit depth - Fermionic: {df['adapt_depth'].mean():.1f}")
    print(f"Mean circuit depth - Qubit: {df['qubit_adapt_depth'].mean():.1f}")

## 9. Summary and Conclusions

In [ ]:
if not results:
    print("No results to summarize.")
else:
    df = pd.DataFrame(results)
    
    print("\n" + "="*80)
    print("FINAL ASSESSMENT")
    print("="*80)
    
    # Check if Qubit-ADAPT achieved the goal
    qubit_wins_cnot = (df['qubit_adapt_cnots'] < df['adapt_cnots']).sum()
    qubit_reaches_chem_acc = (df['qubit_adapt_error'] <= CHEMICAL_ACCURACY).sum()
    
    print(f"\nMolecules tested: {len(df)}")
    print(f"\nCNOT Count Comparison:")
    print(f"  Qubit-ADAPT used fewer CNOTs: {qubit_wins_cnot}/{len(df)} molecules")
    print(f"  Mean CNOT reduction: {((df['adapt_cnots'].mean() - df['qubit_adapt_cnots'].mean()) / max(df['adapt_cnots'].mean(), 1) * 100):.1f}%")
    
    print(f"\nAccuracy Comparison:")
    print(f"  Qubit-ADAPT reached chemical accuracy: {qubit_reaches_chem_acc}/{len(df)} molecules")
    print(f"  Mean error - Fermionic: {df['adapt_error'].mean():.6e} Ha")
    print(f"  Mean error - Qubit: {df['qubit_adapt_error'].mean():.6e} Ha")
    
    print(f"\nCircuit Depth Comparison:")
    print(f"  Mean depth - Fermionic: {df['adapt_depth'].mean():.1f}")
    print(f"  Mean depth - Qubit: {df['qubit_adapt_depth'].mean():.1f}")
    
    print(f"\nParameter Efficiency:")
    print(f"  Mean parameters - Fermionic: {df['adapt_params'].mean():.1f}")
    print(f"  Mean parameters - Qubit: {df['qubit_adapt_params'].mean():.1f}")
    
    # Final verdict
    print(f"\n" + "-"*80)
    if qubit_wins_cnot == len(df) and qubit_reaches_chem_acc > 0:
        print("✅ SUCCESS: Qubit-ADAPT-VQE consistently uses fewer CNOTs while reaching chemical accuracy!")
        print("   This proves its superiority for real quantum hardware deployment.")
    elif qubit_wins_cnot > 0:
        print("⚠️  PARTIAL SUCCESS: Qubit-ADAPT-VQE uses fewer CNOTs for some molecules.")
        print("   Consider adjusting convergence settings or operator pool size.")
    else:
        print("❌ NEEDS INVESTIGATION: Qubit-ADAPT-VQE did not show CNOT advantage.")
        print("   Possible reasons:")
        print("   - Convergence settings too strict")
        print("   - Operator pool size differences")
        print("   - Need more iterations")
    
    print("\n" + "="*80)